<a href="https://colab.research.google.com/github/PETEROA/NAS_LLM/blob/main/NAS_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install gputil

  Preparing metadata (setup.py) ... done
  Created wheel for gputil: filename=GPUtil-1.4.0-py3-none-any.whl size=7392 sha256=169025f350552cc12cca3050510d98bd96422ed6732c317760b49090212f00a0
  Stored in directory: /root/.cache/pip/wheels/92/a8/b7/d8a067c31a74de9ca252bbe53dea5f896faabd25d55f541037
Successfully built gputil


In [ ]:
import torch.nn as nn

In [ ]:
import torch
import torch.nn as nn  # Import torch.nn
import torch.nn.functional as f
from torch import optim
from torch.utils.data import DataLoader, Dataset
import transformers
from transformers import GPT2Tokenizer, GPT2LMHeadModel, get_linear_schedule_with_warmup, AutoTokenizer, AutoModelForCausalLM
import numpy as np
import json
import os
import time
import random
import pickle
import logging
from typing import List, Dict, Tuple, Optional, Any, Union
from dataclasses import dataclass, asdict
from abc import ABC, abstractmethod
import math
import copy
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
import psutil
import GPUtil
import threading
from collections import defaultdict, deque
import heapq
import wandb
import hashlib

In [ ]:
#Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler('nas_training.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

Architecture Config class.. central config blueprint for architecture definition in the NAS framework.. encapsulates all hyperparameters and architectural decisions that define a language model.. includes model dimension, number of layers, vocabulary size, attention heads and feed-forward dimension.. specifies architectural choices including attention type (multi-head, sparse, local, or linear), activation functions(GELU, Swish, RELU, GLU),normalization types(LayerNorm, RMSNorm, ScaleNorm), and position encoding strategies(Sinusoidal, rotary,or ALiBi)... other features include gradient checkpointing, flash attention, and mixture of experts.. hardware constraints are also explicitly tracked

In [ ]:
@dataclass
class ArchitectureConfig:
  # Complete architecture configuration
  vocab_size: int = 50257 #GPT-2 vocab size
  d_model: int = 768
  n_layers: int = 12
  n_heads: int = 12
  d_ff: int = 3072
  max_seq_len: int = 1024

  # Architecture choices
  attention_type: str = "multi_head" # multi_head, sparse, local, linear
  activation: str = "gelu" # gelu, swish, relu, glu\
  norm_type: str = "layer_norm"
  position_encoding: str = "learned"
  attention_dropout: float = 0.1
  residual_dropout: float = 0.1

  #Advanced Features
  use_gradient_checkpointing: bool = False
  use_flash_attention: bool = False
  use_mixture_of_experts: bool = False
  moe_num_experts: int = 8
  moe_top_k: int = 2

  # Hardware Constraints
  max_memory_mb: int = 8000
  max_flops: int = 1000000000
  target_latency_ms: float = 100.0

  def to_dict(self):
    return asdict(self)

  def __hash__(self):
    return hash(str(sorted(self.to_dict().items())))

Class for measuring performance and performance xtics of NN models on target hardware.. profiler provides feedback to the NAS algo about whether candidate architectures are actually deployable given hardware constraints. it maintains a profile cache and detects whether to utilize CUDA or CPU.. main profilling method takes a model and input shape, the executes multiple runs (default 10) to gather reliable stats... memory profiling captures peak memory usage in Megabytes using pytorch memory tracking utilities... profiler also estimates computational cost in FLOPs by anlyzing the model's structure (counting matrix multiplication operations in linear layers and attention mechanisms).

In [ ]:
class HardwareProfiler:
  # Profiles model performance on target hardware
  def __init__(self):
    self.profiles = {}
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  def profile_model(self, model: nn.Module, input_shape: Tuple[int, int],
                  num_runs: int = 10) -> Dict[str, float]:
        # Profile model for memory, FLOPs, and latency
        model.eval()
        model = model.to(self.device)

        # Create dummy input
        dummy_input = torch.randint(0, 1000, input_shape).to(self.device)

        # Warm up
        with torch.no_grad():
            for _ in range(3):
                _ = model(dummy_input)

        # Memory Profilling
        if torch.cuda.is_available():
          torch.cuda.reset_peak_memory_stats()
          torch.cuda.synchronize()

       # Latency profilling
        latencies = []
        for _ in range(num_runs):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start_time = time.time()

            with torch.no_grad():
                _ = model(dummy_input)

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            end_time = time.time()
            latencies.append((end_time - start_time) * 1000)

        # Get memory usage
        memory_mb = 0
        if torch.cuda.is_available():
            memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

        # Estimate FLOPs (simplified)
        flops = self._estimate_flops(model, input_shape)

        return {
            'latency_ms': np.mean(latencies),
            'memory_mb': memory_mb,
            'flops': flops,
            'latency_std': np.std(latencies),
            'params': sum(p.numel() for p in model.parameters())
        }

  def _estimate_flops(self, model: nn.Module, input_shape: Tuple[int, int]) -> int:
        # Estimate FLOPs for transformer model
        batch_size, seq_len = input_shape

        # Rough FLOP estimation for transformer. Simplified calculation
        total_flops = 0

        for module in model.modules():
            if isinstance(module, nn.linear):
              # Matrix Multiplication
              in_features = module.in_features
              out_features = module.out_features
              total_flops += batch_size * seq_len * in_features * out_features
            elif isinstance(module, nn.MultiheadAttention):
              # Attention FLOPs (simplified)
              d_model = module.embed_dim
              total_flops += batch_size * seq_len * d_model

        return total_flops


class implements a flexible attention mechanism module that supports multiple attention variants within a single unified interface.. design allows NAS algo to search over diff attention strategies without changing the overall model structure.... class supports four distinct attention types: standard multi-head attention which computes full pairwise attention scores, sparse attention which randomly masks connections to reduce computational cost, local attention which restricts each token to attend only within a sliding window, and linear attention which approximates attention in linear time time complexity rather than quadratic... Module handles query, key and value projections through a single linear efficiency, then splits them appropraitely.

In [ ]:
class AdvancedAttention(nn.Module):
  # Advanced attention mechanisms with multiple options

  def __init__(self, config: ArchitectureConfig):
    super().__init__()
    self.config = config
    self.attention_type = config.attention_type
    self.d_model = config.d_model
    self.n_heads = config.n_heads
    self.d_head = config.d_model // config.n_heads

    # Query, Key, Value projections
    self.qkv = nn.Linear(config.d_model, 3 * config.d_model, bias=False)
    self.proj = nn.Linear(config.d_model, config.d_model)
    self.dropout = nn.Dropout(config.attention_dropout)

    # Attention specific parameters
    if config.attention_type == "sparse":
      self.sparsity_ratio = 0.1
    elif config.attention_type == "local":
      self.window_size = 64
    elif config.attention_type == "linear":
      self.feature_map = nn.ReLU()

    # Rotary position encoding
    if config.position_encoding == "rotary":
      self.rotary_emb = RotaryEmbedding(self.d_head)

def forward(self, x: torch.Tensor, mask: Optional[torch.tensor] = None) -> torch.Tensor:
  B, T, C = x.shape

  #QKV Projection
  qkv = self.qkv(x)
  q, k, v = qkv.chunk(3, dim=-1)

  # Reshape for multi-head attention
  q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
  k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
  v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

  # Apply position encoding
  if self.config.position_encoding == "rotary":
    q, k = self.rotary_emb(q, k)

  # Attention computation based on type
  if self.attention_type == "multi_head":
    attn_output = self._multi_head_attention(q, k, v, mask)
  elif self.attention_type == "sparse":
    attn_output = self._sparse_attention(q, k, v, mask)
  elif self.attention_type == "local":
    attn_output = self._local_attention(q, k, v, mask)
  elif self.attention_type == "linear":
    attn_output = self._linear_attention(q, k, v, mask)
  else:
    raise ValueError(f"Unknown attention type: {self.attention_type}")

  # Reshape and project
  attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)
  return attn_output

def _multi_head_attention(self, q, k, v, mask):
  # standard multi-head attention
  scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
  if mask is not None:
    scores = scores.masked_fill(mask == 0, -1e9)
  attn_weights = F.softmax(scores, dim=-1)
  attn_weights = self.dropout(attn_weights)
  return torch.matmul(attn_weights, v)

def _sparse_attention(self, q, k, v, mask):
  # Sparse attention simplified
  scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)

  # Apply sparsity mask
  B, H, T, _ = scores.shape
  sparse_mask = torch.rand(B, H, T, T, device=scores.device) < self.sparsity_ratio
  scores = scores.masked_fill(sparse_mask, -1e9)

  if mask is not None:
    scores = scores.masked_fill(mask == 0, -1e9)

  attn_weights = F.softmax(scores, dim=-1)
  attn_weights = self.dropout(attn_weights)
  return torch.matmul(attn_weights, v)

def _local_attention(self, q, k, v, mask):
  # Local attention with sliding window
  B, H, T, D = q.shape

  # Create local attention mask
  local_mask = torch.zeros(B, H, T, T, device=q.device)
  for i in range(T):
    start = max(0, i - self.window_size // 2)
    end = min(T, i + self.window_size // 2 + 1)
    local_mask[:, :, i, start:end] = 1

  scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
  scores = scores.masked_fill(local_mask == 0, -1e9)

  if mask is not None:
    scores = scores.masked_fill(mask == 0, -1e9)

  attn_weights = F.softmax(scores, dim=-1)
  attn_weights = self.dropout(attn_weights)
  return torch.matmul(attn_weights, v)

def _linear_attention(self, q, k, v, mask):
  # Linear attention approximation
  # apply feature map
  q = self.feature_map(q)
  k = self.feature_map(k)

  # linear attention: 0(T * 0) instead of 0(T^2)
  kv = torch.matmul(k.transpose(-2, -1), v)
  attn_output = torch.matmul(q, kv)

  #Normalize
  normalizer = torch.matmul(q, k.sum(dim=-2, keepdim=True).transpose(-2, -1))
  attn_output = attn_output / (normalizer + 1e-6)

  return attn_output


class implements Rotary Position Embeddings (RoPE), a position encoding technique that applies rotational transformations to the query and key vectors in attention mechanisms. Unlike traditional absolute or relative position encodings, RoPE encodes positional information through rotation matrices in high dimensional space, which preserves relative position info and has been shown to improve model performance, especially for longer sequences. The class precomputes inverse frequency values based on the embedding dimension, following the formula inv_freq = 1.0 / (10000^(2i/d)) for dimension pairs. To optimize performance, the module caches cosine and sine values for previously seen sequence lengths, avoiding redundant recomputation. During the forward pass, it generates position-dependent frequencies, computes embeddings by concatenating the frequencies, and then applies the cached cosine and sine values. The core rotation operation splits each vector into two halves and applies the transformation: (x1 * cos - x2 * sin, x1 * sin + x2 * cos), which represents a rotation in 2D subspaces. This rotation naturally encodes the distance between positions and allows the model to extrapolate to longer sequences than it was trained on, making it particularly valuable for language models that need to handle variable-length inputs.

In [ ]:
class RotaryEmbedding(nn.Module):
  # Rotary Position Embedding

  def __init__(self, dim: int, max_seq_len: int = 2048):
    super().__init__()
    self.dim = dim
    inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
    self.register_buffer("inv_freq", inv_freq)

    # Cache for efficiency
    self.max_seq_len = max_seq_len
    self._cos_cached = None
    self._sin_cached = None
    self._seq_len_cached = 0

  def forward(self, q: torch.Tensor, k: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # Apply rotary position embedding to query and key
    seq_len = q.shape[-2] # Correctly get sequence length from the second to last dimension

    if seq_len > self._seq_len_cached or self._cos_cached is None or self._seq_len_cached > self.max_seq_len: # Added max_seq_len check
      self._seq_len_cached = seq_len
      t = torch.arange(seq_len, device=q.device, dtype=self.inv_freq.dtype) # Use seq_len and specify dtype
      freqs = torch.outer(t, self.inv_freq)
      emb = torch.cat((freqs, freqs), dim=-1)
      # Reshape emb for broadcasting with query and key tensors
      emb = emb.view(1, 1, seq_len, self.dim)
      self._cos_cached = emb.cos()
      self._sin_cached = emb.sin()


    return self._apply_rotary_pos_emb(q, self._cos_cached[:, :, :seq_len, :], self._sin_cached[:, :, :seq_len, :]), \
           self._apply_rotary_pos_emb(k, self._cos_cached[:, :, :seq_len, :], self._sin_cached[:, :, :seq_len, :])

  def _apply_rotary_pos_emb(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    # Ensure cos and sin have the same shape as x1 and x2 for element-wise multiplication
    cos = cos.squeeze(1).squeeze(0) # Remove the extra dimensions
    sin = sin.squeeze(1).squeeze(0) # Remove the extra dimensions
    return torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1)

this class implements a conditional computation mechanism where different specialized expert networks process different parts of the input, allowing the model to scale capacity without proportionally increasing computation. This architecture is particularly powerful for NAS as it provides a way to build very large models that remain computationally efficient. The class maintains multiple expert networks (typically 4-16), each implemented as a feed-forward network with configurable activation functions. A gating network (single linear layer) dynamically determines which experts should process each input token. The gating mechanism computes logits for all experts, applies softmax to get probabilities, and then selects the top-k experts (typically k=2) for each token using efficient top-k selection. The selected experts' probabilities are renormalized to sum to 1.0. During forward propagation, the module batches together all tokens assigned to each expert for efficient computation, then combines their outputs weighted by the gate probabilities. This conditional computation pattern means that each token only activates a subset of the total parameters, enabling models to scale to enormous sizes while maintaining reasonable computational costs. The NAS algorithm can search over the number of experts and the top-k value to find optimal configurations for specific tasks and computational budgets.

In [ ]:
class MixtureOfExperts(nn.Module):
  # mixture of experts layer
  def __init__(self, config: ArchitectureConfig):
    super().__init__()
    self.num_experts = config.moe_num_experts
    self.top_k = config.moe_top_k
    self.d_model = config.d_model
    self.d_ff = config.d_ff

    # Gate network
    self.gate = nn.Linear(config.d_model, config.moe_num_experts, bias=False)

    # Expert networks
    self.experts = nn.ModuleList([
        nn.Sequential(
            nn.Linear(config.d_model, config.d_ff),
            self._get_activation(config.activation),
            nn.Linear(config.d_ff, config.d_model)
        ) for _ in range(config.moe_num_experts)
    ])

  def _get_activation(self, activation: str) -> nn.Module:
    if activation == "gelu":
      return nn.GELU()
    elif activation == "swish":
      return nn.SiLU()
    elif activation == "relu":
      return nn.ReLU()
    elif activation == "glu":
      return nn.GLU()
    else:
      return nn.GELU

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    B, T, D = x.shape
    x_flat = x.view(-1, 0)

    # Gate computation
    gate_logits = self.gate(x_flat)
    gate_probs = F.softmax(gate_logits, dim=-1)

    # Select top-k experts
    top_k_probs, top_k_indices = torch.topk(gate_probs, self.top_k, dim=-1)
    top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

    # Expert computation
    output = torch.zeros_like(x_flat)
    for i in range(self.top_k):
      expert_idx = top_k_indices[..., i]
      expert_prob = top_k_probs[:, 1].unsqueeze(-1)

    # Batch expert computation
    expert_inputs = []
    expert_masks = []
    for expert_id in range(self.num_experts):
      mask = expert_idx == expert_id
      if mask.any():
        expert_inputs.append(x_flat[mask])
        expert_masks.append(mask)
      else:
        expert_inputs.append(torch.empty(0, D, device=x.device))
        expert_masks.append(mask)
    for expert_id, (expert_input, mask) in enumerate(zip(expert_inputs, expert_masks)):
      if expert_input.numel() > 0:
        expert_output = self.experts[expert_id](expert_input)
        output[mask] += expert_prob[mask] * expert_output



    return output.view(B, T, D)



AdvancedTransformerBlock class represents a single transformer layer that combines attention and feed-forward components with flexible architectural choices. This is the fundamental building block that gets stacked to create deep transformer models, and its configurability makes it central to the NAS process. The block follows a pre-normalization architecture where layer normalization is applied before (rather than after) the attention and feed-forward sublayers - this design choice has been shown to improve training stability in deep networks. The class instantiates an AdvancedAttention module and either a standard feed-forward network or a MixtureOfExperts layer based on the configuration. It supports multiple normalization types through a factory method that can create LayerNorm, RMSNorm, or ScaleNorm based on the architecture configuration. Similarly, it supports various activation functions including GELU, Swish, ReLU, and GLU. The forward pass implements the classic transformer pattern with residual connections: first normalizing the input, applying attention, adding the residual with dropout, then repeating the process for the feed-forward layer. This architecture can be easily stacked to arbitrary depths, and the NAS algorithm searches over the number of such blocks, their normalization type, activation functions, and whether to use mixture of experts, allowing it to discover optimal depth and width configurations for different tasks.

In [ ]:
class AdvancedTransformerBlock(nn.Module):
  # Advanced Transformer block with multiple architectural choices

  def __init__(self, config: ArchitectureConfig):
    super().__init__()
    self.config = config
    self.attention = AdvancedAttention(config)

    # Feed-Forward or MOE
    if config.use_mixture_of_experts:
      self.ff = MixtureOfExperts(config)
    else:
      self.ff = nn.Sequential(
          nn.Linear(config.d_model, config.d_ff),
          self._get_activation(config.activation),
          nn.Linear(config.d_ff, config.d_model)
      )

    # Normalization
    self.norm1 = self._get_norm(config.norm_type, config.d_model)
    self.norm2 = self._get_norm(config.norm_type, config.d_model)

    # Dropout
    self.dropout = nn.Dropout(config.residual_dropout)

def _get_activation(self, activation: str) -> nn.Module:
  if activation == "gelu":
    return nn.GELU()
  elif activation == "swish":
    return nn.SiLU()
  elif activation == "relu":
    return nn.ReLU()
  elif activation == "glu":
    return nn.GLU()
  else:
    return nn.GELU

def _get_norm(self, norm_type: str, d_model: int) -> nn.Module:
  if norm_type == "layer_norm":
    return nn.LayerNorm(d_model)
  elif norm_type == "rms_norm":
    return nn.RMSNorm(d_model)
  elif norm_type == "scale_norm":
    return nn.ScaleNorm(d_model)
  else:
    return nn.LayerNorm(d_model)

def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
  # Pre-norm architecture
  attn_output = self.attention(self.norm1(x), mask)
  x = x + self.dropout(attn_output)
  ff_output = self.ff(self.norm2(x))
  x = x + self.dropout(ff_output)
  return x


In [ ]:
class RMSNorm(nn.Module):
  # Root Mean Square Layer Normalization
  def __init__(self, d_model: int, eps: float = 1e-8):
    super().__init__()
    self.eps = eps
    self.weight = nn.Parameter(torch.ones(d_model))

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    norm = x.norm(dim=-1, keepdim=True) * (x.size(-1) ** -0.5)
    return self.weight * x / (norm + self.eps)


In [ ]:
class ScaleNorm(nn.Module):
  # Scale Normalization
  def __init__(self, d_model: int, eps: float = 1e-6):
    super().__init__()
    self.eps = eps
    self.scale = nn.Parameter(torch.tensor(d_model ** 0.5))

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    norm = x.norm(dim=-1, keepdim=True)
    return self.scale * x / (norm + self.eps)

The AdvancedLLM class is the complete language model that integrates all the modular components into a functioning transformer-based architecture. This is the actual model that gets instantiated, trained, and evaluated during the NAS process. The class begins by creating token embeddings that map vocabulary indices to dense vectors. Position encoding is handled flexibly based on the configuration: learned embeddings store trainable position vectors, sinusoidal embeddings use fixed trigonometric patterns, ALiBi adds bias terms to attention scores, and rotary embeddings are handled within the attention layers themselves. The core of the model consists of a stack of AdvancedTransformerBlock modules, with the number of blocks determined by the n_layers configuration parameter. The model supports gradient checkpointing as an optional memory-saving technique that trades computation for reduced memory usage by recomputing activations during backpropagation rather than storing them. After processing through all transformer blocks, a final normalization layer is applied, followed by a language modeling head that projects back to vocabulary size for next-token prediction. The forward method orchestrates this pipeline: embedding input tokens, adding positional information, sequentially processing through transformer blocks (with optional checkpointing), applying final normalization, and computing logits over the vocabulary. Weight initialization follows standard practice with small random values (std=0.02) to ensure stable training from the start. This class serves as the evaluation target for the NAS algorithm, which searches for optimal configurations of all its constituent components.

In [ ]:
class AdvancedLLM(nn.Module):
  # Advanced LLM with configurable architecture

  def __init__(self, config: ArchitectureConfig):
    super().__init__()
    self.config = config

    #Embeddings
    self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)

    # Position Encoding
    if config.position_encoding == "learned":
      self.pos_embedding = nn.Embedding(config.max_seq_len, config.d_model)
    elif config.position_encoding == "sinusoidal":
      self.pos_embedding = SinusoidalPositionalEmbedding(config.d_model, config.max_seq_len)
    elif config.position_encoding == "alibi":
      self.pos_embedding = ALiBiPositionalEmbedding(config.n_heads)

    # Rotary is handled in attention

    #transformer blocks
    self.blocks = nn.ModuleList([
        AdvancedTransformerBlock(config) for _ in range(config.num_layers)
    ])

    #Final Norm and projection
    self.ln_f = self._get_norm(config.norm_type, config.d_model)
    self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

    # Gradient checkpointing
    self.gradient_checkpointing = config.use_gradient_checkpointing

    # initialize weights
    self.apply(self._init_weights)

  def _get_norm(self, norm_type: str, d_model: int) -> nn.Module:
    if norm_type == "layer_norm":
      return nn.LayerNorm(d_model)
    elif norm_type == "rms_norm":
      return RMSNorm(d_model)
    elif norm_type == "scale_norm":
      return ScaleNorm(d_model)
    else:
      return nn.LayerNorm(d_model)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    B, T = input_ids.shape
    device = input_ids.device

    # Token embeddings
    x = self.token_embedding(input_ids)

    # Positional embeddings
    if self.config.position_encoding == "learned":
      positions = torch.arrange(T, device=device)
      x = x + self.pos_embedding(positions)
    elif self.config.position_encoding == "sinusoidal":
      x = x + self.pos_embedding(x)
    elif self.config.position_encoding == "alibi":
      # alibi applied in attention
      pass

    for block in self.blocks:
      if self.gradient_checkpointing and self.training:
        x = torch.utils.checkpoint.checkpoint(block, x, attention_mask)
      else:
        x = block(x, attention_mask)

    x = self.ln_f(x)
    logits = self.lm_head(x)

    return logits

In [ ]:
class SinusoidalPositionalEmbedding(nn.Module):
  # Sinusoidal position embedding

  def __init__(self, d_model: int, max_len: int = 5000):
    super().__init__()
    pe = torch.zeros(max_len, d_model)
    position = torch.arrange(0, max_len).unsqueeze(1).float()

    div_term = torch.exp(torch.arrange(0, d_model, 2).float() * - (math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    self.register_buffer('pe', pe.unsqueeze(0))

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.pe[:, :x.size(1)]

In [ ]:
class ALiBiPositionalEmbedding(nn.Module):
  # Attention with linear biases (ALiBi)

  def __init__(self, n_heads: int):
    super().__init__()
    self.n_heads = n_heads
    slopes = self._get_slopes(n_heads)
    self.register_buffer('slopes', slopes)

  def _get_slopes(self, n_heads: int) -> torch.Tensor:
    def get_slopes_power_of_2(n):
      start = (2**(-2**-(math.log2(n)-3)))
      ratio = start
      return [start*ratio**i for i in range(n)]

    if math.log2(n_heads).is_integer():
      return torch.tensor(get_slopes_power_of_2(n_heads))
    else:
      closest_power_of_2 = 2**math.floor(math.log2(n_heads))
      return torch.tensor(get_slopes_power_of_2(closest_power_of_2) + get_slopes_power_of_2(2*closest_power_of_2)[0:n_heads-closest_power_of_2])


  def forward(self, seq_len: int) -> torch.Tensor:
    # Creat position matrix
    positions = torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)
    positions = positions.abs()

    # Apply slopes
    alibi = positions.unsqueeze(0) * self.slopes.unsqueeze(-1).unsqueeze(-1)
    return alibi



The class implements a genetic algorithm approach to neural architecture search, treating architecture configurations as individuals in an evolving population.... this search strategy is particularly effective for discrete search spaces and can escape local optima through mutation and crossover operations. The class defines a comprehensive search space covering all configurable aspects of the architecture: model dimensions (256-1024), layer counts (6-24), attention head counts (4-16), attention types, activation functions, normalization methods, position encodings, mixture of experts settings, and dropout rates. The search process begins by initializing a random population of architectures, with consistency checks ensuring that configuration parameters are compatible (for example, n_heads must divide d_model evenly). The evolution process follows classic genetic algorithm principles: elite selection preserves the top-performing architectures (default 10% of population) directly into the next generation to prevent losing good solutions. The remaining population slots are filled through tournament selection, crossover, and mutation operations. Tournament selection picks parents by running small competitions (typically 3 individuals) and selecting winners based on fitness scores. Crossover creates offspring by randomly selecting each configuration parameter from one of two parents, blending their characteristics. Mutation randomly changes individual parameters with a small probability (default 10%), introducing exploration and preventing premature convergence. The class maintains fitness history across generations, enabling analysis of evolutionary progress. This search strategy is particularly powerful for NAS because it can handle complex, discrete search spaces, naturally maintains diversity in the population, and doesn't require gradient information, making it applicable even when the relationship between architecture and performance is highly non-linear or discontinuous.

In [ ]:
class EvolutionarySearcher:
  # Evolutionary algorithm for architecture search
  def __init__(self, population_size: int = 50, mutation_rate: float = 0.1, crossover_rate: int = 0.8, elite_ratio: float = 0.1):
    self.population_size = population_size
    self.mutation_rate = mutation_rate
    self.crossover_rate = crossover_rate
    self.elite_ratio = int(population_size * elite_ratio)

    # Architecture search space
    self.search_space = {
        'd_model': [256, 384, 512, 768, 1024],
        'n_layers': [6, 8, 12, 16, 20, 24],
        'n_heads': [4, 6, 8, 12, 16,],
        'attention_type': ['multi_head', 'sparse', 'local', 'linear'],
        'activation': ['gelu', 'swish', 'relu'],
        'norm_type': ['layer_norm', 'rms_norm', 'scale_norm'],
        'position_encoding': ['learned', 'sinusoidal', 'alibi', 'rotary'],
        'use_mixture_of_experts': [True, False],
        'moe_num_experts': [4, 8, 16],
        'attention_dropout': [0.0, 0.1, 0.2],
        'residual_dropout': [0.0, 0.1, 0.2],

    }

    self.population = []
    self.fitness_history = []

  def initialize_population(self) -> List[ArchitectureConfig]:
    # Initialize random population
    population = []
    for _ in range(self.population_size):
      config = self._random_config()
      population.append(config)
    return population

  def _random_config(self) -> ArchitectureConfig:
    # Generate random architecture configuration
    config_dict = {}
    for key, values in self.search_space.items():
      config_dict[key] = random.choice(values)

    # Ensure consistency
    if not config_dict['use_mixture_of_experts']:
      config_dict['moe_num_experts'] = 8 # Default value

    # Calculate d_ff based on d_model
    config_dict['d_ff'] = config_dict['d_model'] * 4

    # Ensure n_heads divides d_model
    while config_dict['d_model'] % config_dict['n_heads'] != 0:
      config_dict['n_heads'] = random.choice(self.search_space['n_heads'])

    return ArchitectureConfig(**config_dict)

  def mutate(self, config: ArchitectureConfig) -> ArchitectureConfig:
    # Mutate architecture configuration
    config_dict = config.to_dict()

    for key in self.search_space:
      if random.random() < self.mutation_rate:
        config_dict[key] = random.choice(self.search_space[key])

    # Ensure consistency
    if not config_dict['use_mixture_of_experts']:
      config_dict['moe_num_experts'] = 8

    config_dict['d_ff'] = config_dict['d_model'] * 4

    # Ensure n_heads divides d_model
    while config_dict['d_model'] % config_dict['n_heads'] != 0:
      config_dict['n_heads'] = random.choice(self.search_space['n_heads'])

    return ArchitectureConfig(**config_dict)

  def crossover(self, parent1: ArchitectureConfig, parent2: ArchitectureConfig) -> ArchitectureConfig:
    # Crossover two parent configurations
    config_dict = {}
    parent1_dict = parent1.to_dict()
    parent2_dict = parent2.to_dict()

    for key in self.search_space:
      if random.random() < 0.5:
        config_dict[key] = parent1_dict[key]
      else:
        config_dict[key] = parent2_dict[key]

    # Ensure consistency
    if not config_dict['use_mixture_of_experts']:
      config_dict['moe_num_experts'] = 8

    config_dict['d_ff'] = config_dict['d_model'] * 4

    # Ensure n_heads divides d_model
    while config_dict['d_model'] % config_dict['n_heads'] != 0:
      config_dict['n_heads'] = random.choice(self.search_space['n_heads'])

    return ArchitectureConfig(**config_dict)

  def evolve_generation(self, population: List[ArchitectureConfig], fitness_scores: List[float]) -> List[ArchitectureConfig]:
    # Evolve population for one generation
    # Sort by fitness (higher is better)
    sorted_pairs = sorted(zip(population, fitness_scores), key=lambda x: x[1], reverse=True)
    sorted_population = [pair[0] for pair in sorted_pairs]

    # Elite selection
    new_population = sorted_population[:self.elite_size]

    # Generate offspring
    while len(new_population) < self.population_size:
      if random.random() < self.crossover_rate:
        # Tournament selection
        parent1 = self._tournament_selection(sorted_population, fitness_scores)
        parent2 = self._tournament_selection(sorted_population, fitness_scores)
        child = self.crossover(parent1, parent2)
      else:
        # Mutation only
        parent = self._tournament_selection(sorted_population, fitness_scores)
        child = self.mutate(parent)

      new_population.append(child)

    return new_population[:self.population_size]

  def _tournament_selection(self, population: List[ArchitectureConfig], fitness_scores: List[float], tournament_size: int = 3) -> ArchitectureConfig:
    # Tournament selection
    tournament_indices = random.sample(range(len(population)),min(tournament_size, len(population)))
    tournament_fitness = [fitness_scores[i] for i in tournament_indices]
    winner_idx = tournament_indices[np.argmax(tournament_fitness)]
    return population[winner_idx]


In [ ]:
class ProgressiveSearcher:
  # Progressive NAS with increasing complexity

  def __init__(self, max_stages: int = 4):
    self.max_stages = max_stages
    self.current_stage = 0
    self.stage_configs = self._define_stages()

  def _define_stages(self) -> List[Dict]:
    # Define progressive search stages
    return [
        {
            # stage 1: small models
            'd_model': [256, 384],
            'n_layers': [6, 8],
            'n_heads': [4, 6],
            'max_training_steps': 1000
        },
        {
            # stage 2: meduim models
            'd_model': [384, 512],
            'n_layers': [8, 12],
            'n_heads': [6, 8],
            'max_training_steps': 2000
        },
        {
            # stage 3: large models
            'd_model': [512, 768],
            'n_layers': [12, 16],
            'n_heads': [8, 12],
            'max_training_steps': 3000
        },
        {
            # stage 4: huge models
            'd_model': [768, 1024],
            'n_layers': [16, 24],
            'n_heads': [12, 16],
            'max_training_steps': 5000
        }


    ]
  def get_current_search_space(self) -> Dict:
    # Get current search space
    if self.current_stage < len(self.stage_configs):
      return self.stage_configs[self.current_stage]
    else:
      return self.stage_configs[-1]

  def advance_stage(self):
    # advance to next stage
    if self.current_stage < self.max_stages - 1:
      self.current_stage += 1
      logger.info(f"Advanced to stage {self.current_stage + 1}")


In [ ]:
class HardwareAwareNAS:
  # Hardware aware NAS

  def __init__(self,
               target_device: str = "cuda",
               memory_constraint_mb: float = 8000,
               latency_constraint_ms: float = 100,
               efficiency_weight: float = 0.3):
    self.target_device = target_device
    self.memory_constraint_mb = memory_constraint_mb
    self.latency_constraint_ms = latency_constraint_ms
    self.efficiency_weight = efficiency_weight
    self.profiler = HardwareProfiler()

  def compute_efficiency_score(self, config: ArchitectureConfig, performance_metrics: Dict[str, float]) -> float:
    # Compute hardware efficiency score
    # memory efficiency
    memory_score = max(0, 1 - performance_metrics['memory_mb'] / self.memory_constraint)

    # Latency efficiency
    latency_score = max(0, 1 - performance_metrics['latency_ms'] / self.latency_constraint)

    # Parameter efficiency
    param_score = 1 / (1 + performance_metrics['params'] / 1e6)

    # Combined efficiency score
    efficiency = (memory_score + latency_score + param_score) / 3
    return efficiency

  def is_hardware_feasible(self, performance_metrics: Dict[str, float]) -> bool:
    # Check if hardware constraints are met
    return (performance_metrics['memory_mb'] <= self.memory_constraint and performance_metrics['latency_ms'] <= self.latency_constraint)



In [ ]:
class MultiObjectiveNAS:
  # Multi-objective optimization for NAS

  def __init__(self, objectives: List[str] = ['accuracy', 'efficiency', 'robustness']):
    self.objectives = objectives
    self.pareto_front = []
    self.objective_weights = {obj: 1.0 for obj in objectives}

  def compute_pareto_dominance(self, scores1: Dict[str, float], scores2: Dict[str, float]) -> bool:
    # Check Pareto dominance. Returns 1 if scores1 dominates, -1 if scores2 dominates, 0 if non-dominated
    better_count = 0
    worse_count = 0

    for obj in self.objectives:
      if scores1[obj] > scores2[obj]:
        better_count += 1
      elif scores1[obj] < scores2[obj]:
        worse_count += 1

    if better_count > 0 and worse_count == 0:
      return 1 # scores1 dominates
    elif worse_count > 0 and better_count == 0:
      return -1 # scores2 dominates
    else:
      return 0 # Non-dominated

  def update_pareto_front(self, config: ArchitectureConfig, scores: Dict[str, float]):
    # Update Pareto front with new solution
    dominated_indices = []

    for i, (existing_config, existing_scores) in enumerate(self.pareto_front):
      dominance = self.compute_pareto_dominance(scores, existing_scores)
      if dominance == 1:  # New solution dominates existing
        dominated_indices.append(i)
      elif dominance == -1:  # Existing solution dominates new
        return # dont add new solution

     # Remove dominated solutions
    for i in sorted(dominated_indices, reverse=True):
      del self.pareto_front.pop[i]

    # add new solution
    self.pareto_front.append((config, scores))

    # Add new solution to Pareto front
    self.pareto_front.append((config, scores))

 class provides a PyTorch Dataset interface for loading and preprocessing text data for language modeling tasks, supporting both real datasets from HuggingFace and synthetic data for testing. This class handles the critical but often complex task of preparing training data in a format compatible with the language models being evaluated. The class is initialized with a dataset name (default "wikitext-2"), tokenizer specification (default GPT-2 tokenizer), maximum sequence length (default 1,024 tokens), and dataset split (train/validation). It loads the specified tokenizer from HuggingFace's transformers library and adds a padding token if one doesn't exist (using the EOS token as padding, which is standard practice for GPT-style models). The dataset loading method attempts to fetch the specified dataset from HuggingFace's datasets library, extracting text fields and filtering out empty entries. If loading fails (for example, due to network issues or missing datasets), it falls back to a small set of hardcoded texts about transformers and NAS, repeated 100 times to provide sufficient training data for quick experiments. All texts are tokenized during initialization using the specified tokenizer with truncation to max_length and padding to create uniform-length sequences. This preprocessing is done once and cached, making training more efficient. The getitem method returns (input, target) pairs following the causal language modeling protocol where the input is all tokens except the last, and the target is all tokens except the first (shifted by one position). This dataset abstraction allows the NAS framework to quickly evaluate different architectures on consistent data, and it can be easily extended to support additional datasets or preprocessing strategies specific to particular domains or tasks.

In [ ]:
class Dataset(Dataset):
  # text dataset for language modeling

  def __init__(self, dataset_name: str = "wikitext-2", tokenizer_name: str = "gpt2", max_length: int = 1024, split: str = "train"):
    self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    self.max_length = max_length

    # Add padding token if not present
    if self.tokenizer.pad_token is None:
      self.tokenizer.pad_token = self.tokenizer.eos_token

      # Load dataset
      self.texts = self._load_dataset(dataset_name, split)

      # Tokenize
      self.tokenized_texts = self._tokenize_texts()

  def _load_dataset(self, dataset_name: str, split: str) -> List[str]:
    # Load dataset texts
    try:
      from datasets import load_dataset
      dataset = load_dataset(dataset_name, split=split)
      texts = [item['item'] for item in dataset if item ['text'].strip()]
      logger.info(f"Loaded {len(texts)} texts from {dataset_name}")
      return texts
    except Exception as e:
      logger.warning(f"Could not load {dataset_name}: {e}. Using dummy data")
      return self._create_dummy_texts()

  def _create_dummy_texts(self) -> List[str]:
    # create dummy texts for testing
    texts = [
         "The transformer architecture has revolutionized natural language processing.",
         "Neural architecture search automates the design of deep learning models.",
         "Large language models demonstrate emergent capabilities at scale.",
         "Attention mechanisms allow models to focus on relevant information.",
         "Multi-objective optimization balances multiple competing goals.",
    ] * 100 # Repeat for more data
    return texts

  def _tokenize_texts(self) -> List[torch.Tensor]:
    # Tokenize all texts
    tokenized = []
    for text in tqdm(self.texts, desc="Tokenizing"):
      tokens = self.tokenizer(
          text,
          max_length=self.max_length,
          padding='max_length',
          truncation=True,
          return_tensors='pt'
      )
      tokenized.append(tokens['input_ids'].squeeze(0))
    return tokenized

  def __len__(self) -> int:
    return len(self.tokenized_texts)

  def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
    tokens = self.tokenized_texts[idx]
    # Input is all tokens except last, target is all tokens except first
    input_ids = tokens[:-1]
    labels = tokens[1:]
    return input_ids, labels


The AdvancedNASTrainer class integrates all search strategies and components into a complete Neural Architecture Search system. This is the main entry point that users interact with to run architecture search experiments. The class is initialized with search method selections (evolutionary, progressive, hardware-aware), dataset specification, population size, number of generations, device configuration, output directory, and optional Weights & Biases logging integration. During initialization, it instantiates all the search components: EvolutionarySearcher for genetic algorithm-based search, ProgressiveSearcher for staged complexity increase, HardwareAwareNAS for efficiency optimization, and MultiObjectiveNAS for Pareto front management. It also loads training and validation datasets, creates the output directory, and optionally initializes W&B for experiment tracking. The architecture evaluation method is the core of the trainer - it takes an architecture configuration and training budget, then runs a complete training and evaluation pipeline. First, it checks an evaluation cache to avoid redundant computations (critical for efficiency since the same architecture might be generated multiple times). Then it instantiates the model, profiles its hardware characteristics, checks hardware feasibility, and if feasible, trains the model on the training set and evaluates on validation set. Results include accuracy, perplexity, efficiency score, robustness measure, and hardware metrics. The main search loop runs for a specified number of generations, where each generation involves evaluating all architectures in the population, updating the Pareto front, applying evolution operators, logging metrics, and saving intermediate results. Progressive search stages advance periodically to shift the search space toward larger models. The evolutionary algorithm evolves the population based on fitness scores combining accuracy, perplexity, and efficiency with configurable weights. The search terminates after the maximum number of generations, then saves final results including the best single architecture, complete Pareto front with all non-dominated solutions, search method configurations, and the evaluation cache. This comprehensive framework enables researchers to discover optimal neural architectures for their specific requirements, balancing multiple objectives like accuracy, efficiency, and robustness while respecting real-world hardware constraints.

In [ ]:
class AdvancedNASTrainer:
  # Advanced NAS trainer combining multiple search strategies
  def __init__(self,
               search_methods: List[str] = ['evolutionary', 'progressive', 'hardware_aware'],
               dataset_name: str = "wikitext-2",
               max_generations: int = 20,
               population_size: int = 30,
               device: str = "cuda",
               save_dir: str = "./advanced_nas_results",
               use_wandb: bool = False):

    self.search_methods = search_methods
    self.max_generations = max_generations
    self.device = torch.device(device if torch.cuda.is_available() else "cpu")
    self.save_dir = save_dir
    os.makedirs(save_dir, exist_ok=True)

    #Initialize search components
    self.evolutionary_searcher = EvolutionarySearcher(population_size=population_size)
    self.progressive_searcher = ProgressiveSearcher()
    self.hardware_aware_searcher = HardwareAwareNAS()
    self.multi_objective_nas = MultiObjectiveNAS()

    #data
    self.train_dataset = Dataset(dataset_name=dataset_name, split="train")
    self.val_dataset = Dataset(dataset_name=dataset_name, split="validation")

    # Evaluation Cache
    self.evaluation_cache = {}

    # Results tracking
    self.generation_results = []
    self.best_architectures = []

    #weights and biases
    self.use_wandb = use_wandb
    if use_wandb:
      wandb.init(project="advanced_nas-llm", config={
          'search_methods': search_methods,
          'max_generations': max_generations,
          'population_size': population_size
      })

  def evaluate_architecture(self, config: ArchitectureConfig, max_steps: int = 1000) -> Dict[str, float]:
    # Evaluate architecture performance
    config_hash = str(hash(config))

    # check cache
    if config_hash in self.evaluation_cache:
      return self.evaluation_cache[config_hash]

    logger.info(f"Evaluating architecture: {config_hash}")

    try:
      #create model
      model = AdvancedLLM(config).to(self.device)

      #hardware profiling
      profiler_results = self.hardware_aware_searcher.profiler.profile_model(model, (8, config.max_seq_len // 2))

      #check hardeware feasibility
      if not self.hardware_aware_searcher.is_hardware_feasible(profiler_results):
        logger.warning(f"Hardware constraints not met for architecture: {config_hash}")
        return {
            'accuracy': 0.0,
            'efficiency': 0.0,
            'perplexity': float('inf'),
            'hardware_feasible': False,
            **profiler_results

            }

      # Training evaluation
      training_results = self._train_and_evaluate(model, config, max_steps)

      #compute efficiency score
      efficiency_score = self.hardware_aware_searcher.compute_efficiency_score(config, profiler_results)

      #combine results
      results = {
          'accuracy': training_results.get('accuracy', 0.0),
          'efficiency': efficiency_score,
          'perplexity': training_results.get('perplexity', float('inf')),
          'hardware_feasible': True,
          'robustness': training_results.get('robustness', 0.5),
          **profiler_results
      }

      #cache results
      self.evaluation_cache[config_hash] = results

      return results

    except Exception as e:
      logger.warning(f"Error evaluating architecture: {config_hash}: {e}")
      return {
            'accuracy': 0.0,
            'efficiency':0.0,
            'perplexity': float('inf'),
            'hardware_feasible': False,
            'robustness': 0.0,
            'memory_mb': float('inf'),
            'latency_ms': float('inf'),
            'flops': 0,
            'params':0

        }

  def _train_and_evaluate(self, model: nn.Module, config: ArchitectureConfig, max_steps: int) -> Dict[str, float]:
    # Train and evaluate model
    #data loaders
    train_loader = DataLoader(
        self.train_dataset,
        batch_size=max(1, 32 // max(1, config.d_model // 256)), #Adaptive batch size
        shuffle=True,
        num_workers=0,

    )
    val_loader = DataLoader(
        self.val_dataset,
        batch_size=max(1, 32 // max(1, config.d_model // 256)), #Adaptive batch size
        shuffle=False,
        num_workers=0,
    )

    #optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

    #loss function
    criterion = nn.CrossEntropyLoss(ignore_index=self.train_dataset.tokenizer.pad_token_id)

    #training loop
    model.train()
    total_loss = 0.0
    step = 0

    for batch_index, (input_ids, labels) in enumerate(train_loader):
      if step >= max_steps:
        break

      input_ids = input_ids.to(self.device)
      labels = labels.to(self.device)

      optimizer.zero_grad()

      try:
        logits = model(input_ids)
        loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        step += 1

      except RuntimeError as e:
        if "out of memory" in str(e):
          logger.warning("Out of memory error. Skipping batch.")
          torch.cuda.empty_cache()
          return {'accuracy': 0.0, 'perplexity': float('inf'), 'robustness': 0.0}
        else:
          raise e

      #validation
      model.eval()
      val_loss = 0.0
      val_steps = 0
      correct_predictions = 0
      total_predictions = 0

      with torch.no_grad():
        for batch_idx, (input_ids, labels) in enumerate(val_loader):
          if val_steps >= 50: # limit validation steps
            break

          input_ids = input_ids.to(self.device)
          labels = labels.to(self.device)

          try:
            logits = model(input_ids)
            loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))
            val_loss += loss.item()

            #compute accuracy
            predictions = torch.argmax(logits, dim=-1)
            mask = labels != self.train_dataset.tokenizer.pad_token_id
            correct_predictions += ((predictions == labels) & mask).sum().item()
            total_predictions += mask.sum().item()
            val_steps += 1

          except RuntimeError as e:
            if "out of memory" in str(e):
              torch.cuda.empty_cache()
              break
            else:
              raise e

      #compute metrics
      avg_train_loss = total_loss / max(1, step)
      avg_val_loss = val_loss / max(1, val_steps)
      accuracy = correct_predictions / max(1, total_predictions)
      perplexity = math.exp(min(avg_val_loss,10)) # cap perplexity

      # simple robustness measure (consistency accross batches)
      robustness = max(0.0, 1.0 - (avg_val_loss - avg_train_loss))

      return {
          'accuracy': accuracy,
          'perplexity': perplexity,
          'robustness': robustness,
          'train_loss': avg_train_loss,
          'val_loss': avg_val_loss
      }

  def run_search(self) -> List[Tuple[ArchitectureConfig, Dict[str, float]]]:
    #Run complete NAS process
    logger.info("Starting search")

    #Initialize population
    if 'evolutionary' in self.search_methods:
      population = self.evolutionary_searcher.initialize_population()
    else:
        population = [self.evolutionary_searcher._random_config() for _ in range(self.evolutionary_searcher.population_size)] # Corrected method name

    best_overall_score = -float('inf')
    best_overall_config = None

    for generation in range(self.max_generations):
      logger.info(f"Generation {generation + 1}/{self.max_generations}")

      #Progressive search: adjust search space
      if 'progressive' in self.search_methods and generation % 5 == 0:
        self.progressive_searcher.advance_stage()

      #evaluate population
      generation_results = []
      fitness_scores = []

      for i, config in enumerate(tqdm(population, desc=f"Evaluating Gen {generation + 1}")):
        #Progressive search: limit training steps based on stage
        if 'progressive' in self.search_methods:
          stage_config = self.progressive_searcher.get_current_search_space()
          max_steps = stage_config.get('max_training_steps', 1000)
        else:
          max_steps = 1000

        #Evaluate architecture
        results = self.evaluate_architecture(config, max_steps)

        #compute fitness score (weighted combination)
        if 'hardware_aware' in self.search_methods:
          fitness_score = (0.4 * results['accuracy']) + 0.3 * (1.0 / (1.0 + results['perplexity'])) + 0.3 * results['efficiency']
        else:
          fitness_score = (0.6 * results['accuracy'] + 0.4 * (1.0 / (1.0 + results['perplexity'])))

        fitness_scores.append(fitness_score)
        generation_results.append((config, results))

        #update multi-objective pareto front
        self.multi_objective_nas.update_pareto_front(config,{
            'accuracy': results['accuracy'],
            'efficiency': results['efficiency'],
            'robustness': results['robustness']
        })

        # track best overall
        if fitness_score > best_overall_score:
          best_overall_score = fitness_score
          best_overall_config = config


        #log to wandb
        if self.use_wandb:
          wandb.log({
              'generation': generation,
              'individual': i,
              'fitness_score': fitness_score,
              'accuracy': results['accuracy'],
              'perplexity': results['perplexity'],
              'efficiency': results['efficiency'],
              'memory_mb': results['memory_mb'],
              'latency_ms': results['latency_ms'],
              'params': results['params']
          })

      #store generation results
      self.generation_results.append(generation_results)

      # evolution step
      if 'evolutionary' in self.search_methods and generation < self.max_generations - 1:
        population = self.evolutionary_searcher.evolve_generation(population, fitness_scores) # Corrected method name

      # log generation summary
      avg_fitness = np.mean(fitness_scores) # Corrected variable name
      max_fitness = np.max(fitness_scores)
      logger.info(f"Generation {generation + 1} - Avg Fitness: {avg_fitness:.4f}, Max Fitness, "
                  f"Max Fitness:  {max_fitness:.4f}")

      # save intermediate results
      self._save_generation_results(generation, generation_results, fitness_scores)

    #final results
    logger.info("NAS complete")
    logger.info(f"Best Overall Fitness: {best_overall_score:.4f}")
    logger.info(f"Pareto front size: {len(self.multi_objective_nas.pareto_front)}")

    #save final results
    self._save_final_results(best_overall_config, best_overall_score)

    return self.multi_objective_nas.pareto_front


  def _save_generation_results(self, generation: int, results: List, fitness_scores: List[float]):
    # save generation results
    save_data = {
              'generation': generation,
              'results': [(config.to_dict(), scores) for config, scores in results],
              'fitness_scores': fitness_scores
          }

    with open(os.path.join(self.save_dir, f'generation_{generation}.json'), 'w') as f:
      json.dump(save_data, f, indent=4)

  def _save_final_results(self, best_config: ArchitectureConfig, best_score: float):
    #save final NAS results
    final_results = {'best_architecture': best_config.to_dict() if best_config else None,
            'best_score': best_score,
            'pareto_front': [(config.to_dict(), scores)
                           for config, scores in self.multi_objective_nas.pareto_front],
            'search_methods': self.search_methods,
            'total_evaluations': len(self.evaluation_cache)
         }


    with open(os.path.join(self.save_dir, 'final_results.json'), 'w') as f:
      json.dump(final_results, f, indent=4)

    #save evaluation cache
    with open(os.path.join(self.save_dir, 'evaluation_cache.json'), 'w') as f:
      json.dump(self.evaluation_cache, f, indent=4)

  def visualize_results(results_dir: str):
    """Visualize NAS results"""
    # Load results
    with open(os.path.join(results_dir, 'final_results.json'), 'r') as f:
        results = json.load(f)

    # Extract Pareto front
    pareto_configs = []
    pareto_scores = []
    for config_dict, scores in results['pareto_front']:
        pareto_configs.append(config_dict)
        pareto_scores.append(scores)

    if not pareto_scores:
        logger.warning("No Pareto front solutions found")
        return

    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # Accuracy vs Efficiency
    accuracies = [scores['accuracy'] for scores in pareto_scores]
    efficiencies = [scores['efficiency'] for scores in pareto_scores]

    axes[0, 0].scatter(efficiencies, accuracies, alpha=0.7, s=50)
    axes[0, 0].set_xlabel('Efficiency')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].set_title('Accuracy vs Efficiency (Pareto Front)')
    axes[0, 0].grid(True, alpha=0.3)

    # Architecture distribution
    d_models = [config['d_model'] for config in pareto_configs]
    n_layers = [config['n_layers'] for config in pareto_configs]

    axes[0, 1].scatter(d_models, n_layers, alpha=0.7, s=50)
    axes[0, 1].set_xlabel('Model Dimension')
    axes[0, 1].set_ylabel('Number of Layers')
    axes[0, 1].set_title('Architecture Size Distribution')
    axes[0, 1].grid(True, alpha=0.3)

    # Attention type distribution
    attention_types = [config['attention_type'] for config in pareto_configs]
    attention_counts = {}
    for att_type in attention_types:
        attention_counts[att_type] = attention_counts.get(att_type, 0) + 1

    axes[1, 0].bar(attention_counts.keys(), attention_counts.values())
    axes[1, 0].set_xlabel('Attention Type')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Attention Type Distribution')
    axes[1, 0].tick_params(axis='x', rotation=45)

    # Performance metrics
    metrics = ['accuracy', 'efficiency', 'robustness']
    avg_scores = [np.mean([scores[metric] for scores in pareto_scores]) for metric in metrics]

    axes[1, 1].bar(metrics, avg_scores)
    axes[1, 1].set_ylabel('Average Score')
    axes[1, 1].set_title('Average Performance Metrics')
    axes[1, 1].set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'nas_results_visualization.png'), dpi=300, bbox_inches='tight')
    plt.show()

    # Print summary
    print("\n" + "="*50)
    print("NAS RESULTS SUMMARY")
    print("="*50)
    print(f"Total Pareto-optimal architectures: {len(pareto_configs)}")
    print(f"Best accuracy: {max(accuracies):.4f}")
    print(f"Best efficiency: {max(efficiencies):.4f}")
    print(f"Average accuracy: {np.mean(accuracies):.4f}")
    print(f"Average efficiency: {np.mean(efficiencies):.4f}")

def main():
    """Main function to run advanced NAS"""
    # Configuration
    config = {
        'search_methods': ['evolutionary', 'progressive', 'hardware_aware'],
        'dataset_name': 'wikitext-2',
        'max_generations': 10,  # Reduced for demo
        'population_size': 20,  # Reduced for demo
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'save_dir': './advanced_nas_results',
        'use_wandb': False  # Set to True to use Weights & Biases
    }

    logger.info("Advanced NAS Configuration:")
    logger.info(json.dumps(config, indent=2))

    # Initialize trainer
    trainer = AdvancedNASTrainer(**config)

    # Run search
    pareto_front = trainer.run_search()

    # Visualize results
    trainer.visualize_results(config['save_dir'])

    logger.info("Advanced NAS completed successfully!")

if __name__ == "__main__":
    main()

Evaluating Gen 1: 100%|██████████| 20/20 [00:05<00:00,  3.55it/s]


AttributeError: 'EvolutionarySearcher' object has no attribute 'elite_size'